# 🦜 VieNeu-Audio (Colab T4)

**Trước tiên:** Runtime → Change runtime type → **T4 GPU** → Save. (Backbone không lượng tử hoá — cần GPU thật để render nhanh và để GPU-batch hoạt động.)

Chạy lần lượt từng cell. Nếu mất kết nối giữa chừng, xem mục cuối cùng.

## 1. Gắn Google Drive
Code và output đều lưu ở đây — không mất khi mất kết nối.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/VieNeu-Audio'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print('Đã có sẵn:', os.listdir(PROJECT_DIR))

## 2. Cấu hình Gemini API Key (bắt buộc)
Agent Alpha (tách chương) dùng Gemini API — **bắt buộc phải có key thì Batch mới chạy được**, kể cả file .txt đơn giản. Lấy key miễn phí tại [aistudio.google.com/apikey](https://aistudio.google.com/apikey).

Cách an toàn nhất trên Colab: **🔑 (icon chìa khoá) ở sidebar bên trái → Add new secret → Name: `GEMINI_API_KEY` → dán key vào Value → bật "Notebook access"**. Cell dưới sẽ tự đọc từ đó.

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["GEMINI_API_KEY"] = getpass("Chưa tìm thấy secret GEMINI_API_KEY — dán key trực tiếp vào đây: ")

print("Đã cấu hình GEMINI_API_KEY." if os.environ.get("GEMINI_API_KEY") else "❌ CHƯA có GEMINI_API_KEY — Batch sẽ lỗi ở bước Agent Alpha.")

## 3. Upload code
Chỉ cần lần đầu, hoặc khi có bản code mới. Nếu `pipeline/` và `voxdirector/` đã có sẵn ở Bước 1, bỏ qua cell này. **Zip phải chứa cả 2 thư mục `pipeline/` và `voxdirector/`** (voxdirector/ chứa 4 Agent — Alpha/Beta/Gamma/Delta — không có nó Batch sẽ báo lỗi ngay ở bước Alpha).

In [ ]:
from google.colab import files
import zipfile, io

uploaded = files.upload()
zip_name = next(iter(uploaded))
with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as zf:
    zf.extractall(PROJECT_DIR)
print(os.listdir(PROJECT_DIR))

## 4. Cài dependencies
ffmpeg, font tiếng Việt, vieneu + neucodec (codec GPU cho backbone không lượng tử hoá) + transformers/accelerate (vieneu[gpu] không tự kéo 2 gói này trên Linux), gradio, python-docx. Upgrade torchao/torchtune để tránh xung đột phiên bản trên Colab. Thêm các gói cho voxdirector/ (4 Agent): google-genai, langgraph, chromadb, sentence-transformers, faster-whisper, jiwer.

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg fonts-noto
!fc-cache -f
!pip install -q -U torchao torchtune neucodec
!pip install -q vieneu transformers accelerate python-docx gradio soundfile "numpy<2.1" "requests==2.32.4"
!pip install -q google-genai langgraph chromadb sentence-transformers faster-whisper jiwer

## 5. Kiểm tra nhanh
Cần thấy `h264_nvenc`/`libx264` (encoder) và ít nhất 1 dòng font "Noto Sans". Thiếu font → phụ đề tiếng Việt sẽ lỗi thành ô vuông.

In [ ]:
!ffmpeg -hide_banner -encoders 2>/dev/null | grep -E "nvenc|qsv|libx264"
!fc-list | grep -i "noto sans" | head -3

## 6. Khởi chạy
In ra link `https://xxxxx.gradio.live` — mở link đó để dùng: **① Chọn giọng → ② Nghe mẫu → ③ Render → Video (Batch)**.

An toàn để chạy lại cell này nhiều lần (miễn Bước 1, 2 đã chạy trong phiên hiện tại).

In [ ]:
import sys

try:
    PROJECT_DIR
except NameError:
    raise RuntimeError("Chạy lại Bước 1 (Gắn Drive) và Bước 2 (API key) trước — runtime vừa được cấp phát lại.")

try:
    app.close()
except Exception:
    pass
for mod in list(sys.modules):
    if mod == "pipeline" or mod.startswith("pipeline.") or mod == "voxdirector" or mod.startswith("voxdirector."):
        del sys.modules[mod]
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from pipeline.auto_tts import app
app.queue().launch(share=True, debug=True)

## 🔁 Mất kết nối?

Bình thường trên Colab free (tab rảnh ~90 phút, hoặc phiên quá ~12 tiếng) — không mất phần đã render, chỉ cần chạy lại:

1. Connect lại.
2. Chạy lại Bước 1, 2 (bắt buộc — máy ảo mới hoàn toàn, kể cả biến môi trường API key).
3. Bỏ qua Bước 3 (code đã ở trong Drive).
4. Chạy lại Bước 4, 6.
5. Mở link Gradio mới, upload lại đúng các file đang xử lý dở, chạy tiếp — chương/phần đã xong sẽ tự bỏ qua.